# MAI-Transcribe-2
## Feature demo: from conversation to usable transcript

An approximately 8-10 minute, **file-based** demo: listen to a recording, transcribe it, inspect speakers and word timings, compare output styles and keyword hints, and try Dutch with automatic language detection.

**Before running:** follow the [demo instructions](demo-instructions.md), configure an Azure Speech/Foundry resource with access to MAI-Transcribe-2, and supply a short recording you have permission to process. No audio or credentials are bundled. Use 30-90 second mono PCM WAV files for the full experience; the transcription helper also accepts MP3 and FLAC.

**Cost and privacy:** Run All makes three paid requests for the main recording when comparisons are enabled, plus one if you configure the Dutch sample. There are no automatic retries or model substitutions. Audio is sent to Azure. Notebook outputs may contain sensitive transcripts: clear outputs before sharing or committing. Export is off by default.

This notebook uses `MAI-Transcribe-2`, not deprecated version 1 or the separate streaming model. See the [extended information](information.md) for dated pricing, supported regions, documentation conflicts, and streaming considerations.

## 1. Setup and configuration

In Codespaces or the dev container, select **Python (MAI demos)**; dependencies are preinstalled. For a local setup, select the Python kernel from your `.venv`. If imports fail, install `requirements.txt` into that kernel's environment and restart it.

In Codespaces, supply `SPEECH_ENDPOINT` and `SPEECH_KEY` through [Codespaces secrets](demo-instructions.md#github-codespaces); no `.env` is needed. For local configuration, copy `.env.example` to `.env` beside the notebook, fill in the settings, and save. After editing `.env`, simply rerun the configuration cell: **no kernel or server restart is needed**. Settings are read fresh from the working directory without modifying the process environment. Existing environment variables take precedence; no parent directories are searched. Changing Codespaces secrets requires stopping and restarting the codespace.

Set `SPEECH_ENDPOINT` to your HTTPS resource root, for example `https://your-resource.cognitiveservices.azure.com`. Authentication can use `SPEECH_KEY` from `.env` or the environment, a hidden prompt (`AUTH_MODE = "key"`), or Azure CLI credentials (`AUTH_MODE = "entra"`, after `az login`). Entra authentication requires an appropriate resource role, such as **Cognitive Services Speech User**. Do not put keys in a cell. The `.env` file is ignored by Git.

The `samples/` and `outputs/` folders are ignored by Git. Relative audio paths below are relative to this notebook's working directory.

In [ ]:
import copy
import getpass
import io
import json
import math
import os
import time
import wave
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit
from uuid import uuid4

import pandas as pd
import requests
from dotenv import dotenv_values
from IPython.display import Audio, FileLink, display


In [ ]:
SETTINGS = {**dotenv_values(Path.cwd() / ".env"), **os.environ}

SPEECH_ENDPOINT = (SETTINGS.get("SPEECH_ENDPOINT") or "").strip()
AUTH_MODE = "key" if (SETTINGS.get("SPEECH_KEY") or "").strip() else "entra"
AUDIO_PATH = Path(SETTINGS.get("AUDIO_FILE") or "samples/meeting-en.wav")
dutch_sample = Path.cwd() / "samples/meeting-nl.wav"
DUTCH_AUDIO_PATH = dutch_sample if dutch_sample.is_file() else None  # Set None to skip.

RUN_COMPARISONS = True  # Two additional requests: clean output and keyword hints.
EXPORT_RESULTS = False  # Opt in only if you want local audio-derived data saved.
KEYWORDS = ["Microsoft Foundry", "MAI-Transcribe", "The A Bit Of AI Show"]

MODEL = "MAI-Transcribe-2"
API_VERSION = "2025-10-15"
MAX_DEMO_BYTES = 20 * 1024 * 1024  # Demo guardrail, not the model's service limit.
_prompted_key = None
_credential = None
print(f"Model: {MODEL} | Authentication: {AUTH_MODE} | Main recording: {AUDIO_PATH}")
print("Main recording requests:", 3 if RUN_COMPARISONS else 1)
if not SPEECH_ENDPOINT:
    print("SETUP REQUIRED: fill in SPEECH_ENDPOINT in .env, save, and rerun this cell.")
if not AUDIO_PATH.expanduser().is_file():
    print("SETUP REQUIRED: add a recording and set AUDIO_FILE in .env, then rerun this cell.")


### Small API helper

The following cell contains the plumbing so the presentation cells stay short. It validates inputs before contacting Azure, sets explicit timeouts, rejects redirects, reports errors without dumping request headers, and preserves the actual model response.

The endpoint must be an Azure resource root, not a Foundry project URL. The helper deliberately permits only the public Azure `cognitiveservices.azure.com` host suffix used in this demo. Sovereign-cloud endpoints require a reviewed configuration change.

In [ ]:
def resource_endpoint(value):
    parsed = urlsplit(value)
    hostname = parsed.hostname or ""
    suffix = ".cognitiveservices.azure.com"
    if (parsed.scheme != "https" or not hostname.endswith(suffix)
            or not hostname[:-len(suffix)] or parsed.username or parsed.password
            or parsed.port not in (None, 443) or parsed.path not in ("", "/")
            or parsed.query or parsed.fragment):
        raise ValueError("Set SPEECH_ENDPOINT to https://<resource>.cognitiveservices.azure.com")
    return value.rstrip("/")


def validate_audio(path):
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Recording not found: {path}. Add your audio and update AUDIO_PATH.")
    if path.suffix.lower() not in {".wav", ".mp3", ".flac"}:
        raise ValueError("Use a WAV, MP3, or FLAC recording.")
    if not 0 < path.stat().st_size <= MAX_DEMO_BYTES:
        raise ValueError("Use a nonempty recording no larger than the demo's 20 MiB limit.")
    if path.suffix.lower() == ".wav":
        try:
            with wave.open(str(path), "rb") as recording:
                if recording.getnchannels() != 1 or recording.getsampwidth() != 2:
                    raise ValueError("For this demo, use a mono, 16-bit PCM WAV recording.")
                if recording.getnframes() == 0:
                    raise ValueError("The WAV recording has no audio frames.")
        except (wave.Error, EOFError) as exc:
            raise ValueError("Cannot read this WAV. Export it as mono, 16-bit PCM WAV.") from exc
    return path


def auth_headers():
    global _prompted_key, _credential
    if AUTH_MODE == "key":
        key = (SETTINGS.get("SPEECH_KEY") or "").strip() or _prompted_key
        if not key:
            key = getpass.getpass("Speech resource key (hidden): ").strip()
            if not key:
                raise ValueError("No key supplied. Set SPEECH_KEY or use AUTH_MODE = 'entra'.")
            _prompted_key = key
        return {"Ocp-Apim-Subscription-Key": key}
    if AUTH_MODE == "entra":
        from azure.identity import AzureCliCredential
        if _credential is None:
            _credential = AzureCliCredential()
        token = _credential.get_token("https://cognitiveservices.azure.com/.default")
        return {"Authorization": f"Bearer {token.token}"}
    raise ValueError("AUTH_MODE must be 'key' or 'entra'.")


def request_definition(style="verbatim", phrases=None, diarization=True):
    if style not in {"verbatim", "clean"}:
        raise ValueError("Style must be 'verbatim' or 'clean'.")
    definition = {
        "enhancedMode": {
            "enabled": True,
            "model": MODEL,
            "modelOptions": {"timestamps": "word", "transcribeStyle": style},
        },
        "diarization": {"enabled": diarization},
    }
    if phrases:
        if not all(isinstance(term, str) and term.strip() for term in phrases):
            raise ValueError("Keyword hints must be nonempty strings.")
        definition["phraseList"] = {"phrases": list(phrases)}
    return definition


def validate_result(payload):
    if not isinstance(payload, dict):
        raise ValueError("Expected a JSON object from the transcription service.")
    combined = payload.get("combinedPhrases")
    if not isinstance(combined, list) or not all(
        isinstance(item, dict) and isinstance(item.get("text"), str) for item in combined
    ):
        raise ValueError("Response is missing a valid combinedPhrases transcript list.")
    segments = payload.get("phrases", [])
    if not isinstance(segments, list):
        raise ValueError("Response phrases must be a list.")
    duration = payload.get("durationMilliseconds")
    if duration is not None and not valid_time(duration):
        raise ValueError("Response audio duration must be a nonnegative finite number.")
    for segment in segments:
        if not isinstance(segment, dict) or not isinstance(segment.get("text"), str):
            raise ValueError("A transcript segment is missing text.")
        words = segment.get("words", [])
        if not isinstance(words, list) or not all(
            isinstance(word, dict) and isinstance(word.get("text"), str) for word in words
        ):
            raise ValueError("A transcript segment contains invalid word metadata.")
        for timed_item in [segment, *words]:
            for field in ("offsetMilliseconds", "durationMilliseconds"):
                if field in timed_item and not valid_time(timed_item[field]):
                    raise ValueError(f"Invalid {field} in transcript metadata.")
    return payload


def valid_time(value):
    return (isinstance(value, (int, float)) and not isinstance(value, bool)
            and math.isfinite(value) and value >= 0)


@dataclass
class TranscriptionRun:
    audio_path: Path
    definition: dict
    response: dict
    elapsed_seconds: float

    @property
    def text(self):
        return "\n".join(item["text"] for item in self.response["combinedPhrases"])


def transcribe(path, definition):
    endpoint = resource_endpoint(SPEECH_ENDPOINT)
    path = validate_audio(path)
    headers = auth_headers()
    url = f"{endpoint}/speechtotext/transcriptions:transcribe"
    started = time.perf_counter()
    try:
        with path.open("rb") as audio:
            with requests.post(
                url, params={"api-version": API_VERSION}, headers=headers,
                files={"audio": (path.name, audio, "application/octet-stream")},
                data={"definition": json.dumps(definition)},
                timeout=(10, 180), allow_redirects=False,
            ) as response:
                if response.status_code != 200:
                    guidance = {
                        400: "Check the audio format and model-specific request options.",
                        401: "Check the resource key or sign in again with az login.",
                        403: "Check resource permissions and model access.",
                        404: "Check the resource endpoint, region, and model availability.",
                        413: "The service rejected the recording size; use a shorter clip.",
                        429: "The resource is throttled; wait before manually retrying.",
                    }.get(response.status_code, "Check service health and endpoint configuration before retrying.")
                    raise RuntimeError(f"Azure transcription returned HTTP {response.status_code}. {guidance} No automatic retry was made.")
                try:
                    payload = response.json()
                except ValueError:
                    raise RuntimeError("Azure returned HTTP 200 but the response was not valid JSON.") from None
    except requests.RequestException as exc:
        raise RuntimeError(f"Transcription request failed ({type(exc).__name__}). Check connectivity. No automatic retry was made; a timed-out request may still have been billed.") from None
    return TranscriptionRun(path, copy.deepcopy(definition), validate_result(payload), time.perf_counter() - started)


def transcript_tables(run):
    segment_rows, word_rows = [], []
    for index, segment in enumerate(run.response.get("phrases", [])):
        identity = {"segment": index, "speaker": segment.get("speaker"), "language": segment.get("locale")}
        def row(item):
            start = item.get("offsetMilliseconds")
            duration = item.get("durationMilliseconds")
            return {**identity, "start_s": None if start is None else start / 1000,
                    "duration_s": None if duration is None else duration / 1000,
                    "text": item["text"]}
        segment_rows.append(row(segment))
        word_rows.extend(row(word) for word in segment.get("words", []))
    columns = ["segment", "speaker", "language", "start_s", "duration_s", "text"]
    return pd.DataFrame(segment_rows, columns=columns), pd.DataFrame(word_rows, columns=columns)


def show_run(run):
    duration = run.response.get("durationMilliseconds")
    audio_seconds = None if duration is None else duration / 1000
    display(pd.DataFrame([{
        "model": run.definition["enhancedMode"]["model"],
        "audio_seconds": audio_seconds,
        "request_seconds": round(run.elapsed_seconds, 3),
    }]))
    print(run.text if run.text.strip() else "No speech was returned by the service.")


def wav_excerpt(path, start_seconds, end_seconds):
    path = validate_audio(path)
    if path.suffix.lower() != ".wav":
        raise ValueError("Word playback requires a mono PCM WAV file. MP3/FLAC still support transcription tables.")
    if not valid_time(start_seconds) or not valid_time(end_seconds) or end_seconds <= start_seconds:
        raise ValueError("Choose a finite playback interval with 0 <= start < end.")
    with wave.open(str(path), "rb") as source:
        rate = source.getframerate()
        start_frame = int(start_seconds * rate)
        end_frame = min(source.getnframes(), int(end_seconds * rate))
        if start_frame >= end_frame:
            raise ValueError("Playback interval falls outside this recording.")
        source.setpos(start_frame)
        frames = source.readframes(end_frame - start_frame)
        buffer = io.BytesIO()
        with wave.open(buffer, "wb") as output:
            output.setnchannels(source.getnchannels())
            output.setsampwidth(source.getsampwidth())
            output.setframerate(rate)
            output.writeframes(frames)
    return buffer.getvalue()


def export_run(run, root=Path("outputs")):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    directory = root / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8])
    directory.mkdir()
    (directory / "transcript.txt").write_text(run.text, encoding="utf-8")
    (directory / "response.json").write_text(json.dumps(run.response, ensure_ascii=False, indent=2), encoding="utf-8")
    metadata = {"model": run.definition["enhancedMode"]["model"], "request_definition": run.definition,
                "elapsed_seconds": run.elapsed_seconds, "exported_at": datetime.now(timezone.utc).isoformat()}
    (directory / "run.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    return directory


## 2. Listen first

Use a two-person fictional project update with a few natural fillers, a correction, and names such as Contoso or Microsoft Foundry. See the original sample script in the README. Listen for words you expect the model to find challenging.

This cell checks the endpoint and local file but does **not** contact Azure. Audio playback embeds the recording in notebook output, so clear outputs before sharing.

In [ ]:
_ = resource_endpoint(SPEECH_ENDPOINT)
_ = validate_audio(AUDIO_PATH)
display(Audio(filename=str(AUDIO_PATH.expanduser())))


## 3. Transcribe: one model, one request

The baseline explicitly selects MAI-Transcribe-2, requests word timestamps and speaker diarization, and uses **verbatim** output. No language is forced. No keyword hints are supplied yet.

**Running the next cell sends the recording to Azure and incurs usage charges.** Request time includes upload and response handling, not credential acquisition. The audio duration reported by the service is a separate measurement.

In [ ]:
baseline_definition = request_definition()
print(json.dumps(baseline_definition, indent=2))
baseline = None  # Do not leave an older result looking current if a rerun fails.
baseline = transcribe(AUDIO_PATH, baseline_definition)
show_run(baseline)


## 4. Who spoke, and when?

Speaker IDs are anonymous labels within this recording, **not identities**. Missing timing or speaker fields remain missing; we do not invent them. The response's `confidence` field is not shown as an accuracy percentage because its model-specific calibration is not established.

Compare the segment table with the audio, then inspect individual words. Full tables remain available in `segments` and `words`.

In [ ]:
segments, words = transcript_tables(baseline)
print(f"{len(segments)} segments; {len(words)} timed-word entries")
if segments.empty:
    print("No segment metadata was returned for this request.")
else:
    display(segments.head(30))
if words.empty:
    print("No word metadata was returned. Check the model options and actual response.")
else:
    display(words.head(40))


### Navigate the recording by word

For a PCM WAV input, select a word index to create a short playable excerpt with 0.35 seconds of surrounding context. This uses the timestamps already returned; it makes **no new Azure calls**. Click Play in the audio control.

Widgets need JupyterLab's widget support, or the Jupyter extension in VS Code. If widgets are unavailable, use the table and call `display(Audio(data=wav_excerpt(AUDIO_PATH, 1.0, 3.0)))` with your chosen start/end seconds instead.

In [ ]:
if baseline.audio_path.suffix.lower() != ".wav":
    print("Word playback is WAV-only; use the timestamp tables for this MP3/FLAC recording.")
elif words.dropna(subset=["start_s", "duration_s"]).empty:
    print("Word playback unavailable: the response has no usable word timings.")
else:
    import ipywidgets as widgets
    timed_words = words.dropna(subset=["start_s", "duration_s"]).reset_index(drop=True)
    def play_word(index):
        word = timed_words.iloc[index]
        start = max(0.0, float(word.start_s) - 0.35)
        end = float(word.start_s + word.duration_s) + 0.35
        print(f"{word.text} | speaker {word.speaker} | starts at {word.start_s:.3f}s")
        display(Audio(data=wav_excerpt(baseline.audio_path, start, end)))
    word_picker = widgets.IntSlider(value=0, min=0, max=len(timed_words) - 1,
                                    description="Word index", continuous_update=False)
    display(widgets.VBox([word_picker, widgets.interactive_output(play_word, {"index": word_picker})]))


## 5. Verbatim versus clean

Change only the output style on the **same audio**. Clean output is intended to remove fillers and improve readability; it is not a summary. Neither style guarantees a word-perfect record.

The next cell makes **one additional paid request** when `RUN_COMPARISONS` is true. If the recording contains no disfluencies, the outputs may legitimately be similar.

In [ ]:
clean = None
if RUN_COMPARISONS:
    clean_definition = copy.deepcopy(baseline.definition)
    clean_definition["enhancedMode"]["modelOptions"]["transcribeStyle"] = "clean"
    clean = transcribe(baseline.audio_path, clean_definition)
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame({"Verbatim": [baseline.text], "Clean": [clean.text]}))
    print(f"Clean request: {clean.elapsed_seconds:.3f} seconds")
else:
    print("Style comparison skipped: RUN_COMPARISONS is False.")


## 6. Help with specialized vocabulary

Compare the baseline with a second **verbatim** run using `phraseList.phrases`. Keep the same audio and other options so you are comparing one setting at a time. Edit `KEYWORDS` to match words actually present in your recording.

Hints are not forced output, and improvement is not guaranteed. A correct baseline may need no improvement. This is **one additional paid request** when comparisons are enabled.

In [ ]:
hinted = None
if RUN_COMPARISONS:
    if not KEYWORDS or not all(isinstance(term, str) and term.strip() for term in KEYWORDS):
        raise ValueError("Set KEYWORDS to a nonempty list of terms spoken in the recording.")
    hinted_definition = copy.deepcopy(baseline.definition)
    hinted_definition["phraseList"] = {"phrases": list(KEYWORDS)}
    print("Keyword hints:", KEYWORDS)
    hinted = transcribe(baseline.audio_path, hinted_definition)
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame({"Without hints": [baseline.text], "With hints": [hinted.text]}))
    print(f"Keyword-hinted request: {hinted.elapsed_seconds:.3f} seconds")
else:
    print("Keyword comparison skipped: RUN_COMPARISONS is False.")


## 7. Optional: Dutch, without selecting a language

The configuration cell automatically selects `samples/meeting-nl.wav` when present. Set `DUTCH_AUDIO_PATH = None` to skip, or choose a different recording. Rerun configuration after adding a file. This section makes **one additional paid request** when a recording is selected. Automatic language detection remains enabled because no `locales` hint is sent.

Dutch is one of the model's 60 documented languages. Transcription is not translation: expect Dutch text, not English. Mixed-language performance should be rehearsed, not assumed.

In [ ]:
dutch = None
if DUTCH_AUDIO_PATH is None:
    print("Dutch demo skipped. Set DUTCH_AUDIO_PATH to a recording to enable it.")
else:
    dutch_path = validate_audio(DUTCH_AUDIO_PATH)
    display(Audio(filename=str(dutch_path)))
    dutch = transcribe(dutch_path, request_definition())
    show_run(dutch)
    dutch_segments, _ = transcript_tables(dutch)
    display(dutch_segments.head(20))


## 8. Inspect and optionally export

The parsed JSON response is available as `baseline.response`; the sanitized request configuration is `baseline.definition`. The raw response may contain private speech content, so inspect it only when appropriate for your audience.

Set `EXPORT_RESULTS = True` to save the baseline transcript, full response, and request/timing metadata under a new unique `outputs/` directory. Existing runs are never overwritten. No audio, credentials, or endpoint is included in the export; transcripts and phrase hints can still be sensitive.

In [ ]:
if EXPORT_RESULTS:
    export_directory = export_run(baseline)
    print("Saved baseline results to", export_directory)
    for name in ("transcript.txt", "response.json", "run.json"):
        display(FileLink(str(export_directory / name)))
else:
    print("Export disabled. Results remain in kernel memory and displayed notebook output.")


## Presenter wrap-up

- One Microsoft-built model transcribes the recording and returns metadata useful for navigation and speaker separation.
- Clean/verbatim style and keyword hints serve different purposes; compare them independently.
- Language detection does not translate the audio, and diarization does not identify people.
- Timings shown here are your actual end-to-end request measurements, not a benchmark promise.
- A summary or action-item generator would be a separate downstream model step.

**Optional next demo:** MAI-Transcribe-2-Streaming uses a separate preview model and integration. A local notebook can access a local microphone; a hosted kernel cannot directly access your laptop microphone. Do not represent this file-upload notebook as live streaming.

### Troubleshooting

| Problem | Check |
| --- | --- |
| File not found | Notebook working directory, `AUDIO_PATH`, and the sample file name |
| Invalid WAV | Export mono, 16-bit PCM WAV; renaming a compressed file does not convert it |
| `az` / Entra authentication error | Azure CLI installation, `az login`, subscription, and Speech resource permissions; or choose hidden key authentication |
| HTTP 401 / 403 | Matching resource credentials, role assignment, and model access |
| HTTP 400 / 404 | Explicit model selection, API version, supported region, endpoint, and audio format |
| HTTP 429 / timeout | Wait and inspect resource/service health before manually retrying; the original call may still be billable |
| No words or speakers | Inspect the real response and requested options; do not fabricate missing metadata |
| No widget output | Enable widgets or use the plain timestamp table and manual WAV excerpt helper |

**Before sharing:** use Restart Kernel and Clear All Outputs, save the notebook, and inspect your Git diff. Clearing outputs removes embedded audio and transcript displays; restarting the kernel also releases in-memory credentials and results.

### References

- [MAI-Transcribe model guide](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-transcribe)
- [Speech service regions](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/regions?tabs=llmspeech)
- [Speech role-based access control](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/role-based-access-control)
- [Streaming model overview](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-transcribe-2-streaming)
- [Extended model information](information.md)
